# Predicting Food Delivery Time

## Team Members

- Ebru Özcan, ozcane21@itu.edu.tr, 090210357
- Zeynep Aslı Üretme, uretme20@itu.edu.tr, 090200336

## Libraries

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error,root_mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import cross_val_score


## Data Set Introduction

Technical Information and Motivation:

The dataset used in this project, Food_Delivery_Times.csv, provides historical data on food deliveries. It includes a variety of features that influence delivery times, such as distance, preparation time, traffic level, weather conditions, and courier experience. This dataset is highly relevant for understanding the dynamics of food delivery logistics and optimizing operations through predictive modeling.

Dataset Overview:

Numerical Variables:

Distance_km: The distance covered during delivery (in kilometers).

Preparation_Time_min: The time taken to prepare the food order (in minutes).

Courier_Experience_yrs: The years of experience of the courier.

Delivery_Time_min: The target variable representing delivery time (in minutes).

Categorical Variables:

Weather: Describes weather conditions (e.g., Sunny, Rainy).

Traffic_Level: Indicates traffic levels (e.g., Low, Medium, High).

Time_of_Day: Specifies the time of day (e.g., Morning, Afternoon).

Vehicle_Type: Represents the type of vehicle used for delivery (e.g., Scooter, Car).

Engineered Features:

Preparation_Ratio: The ratio of preparation time to delivery time.

Traffic_Vehicle: A combined feature capturing the interaction between traffic levels and vehicle types.

Missing Values:

Categorical Variables: Weather, Traffic_Level, and Time_of_Day had missing values imputed using the most frequent category.

Numerical Variable: Courier_Experience_yrs had missing values imputed with the mean.

Dataset Source:

This dataset is either derived from operational data or sourced from a publicly available repository. Proper references will be provided based on the exact source.



## Description of the Problem

*Research Questions:*

The project focuses on addressing several key research questions related to predicting delivery times and improving operational efficiency in food delivery services:

1. *Regression Problem:*  
   - Predict the delivery time (Delivery_Time_min) using a combination of numerical and categorical features such as distance, preparation time, traffic level, weather conditions, and vehicle type.  
   - Evaluate and compare the performance of different regression models, including Linear Regression, LightGBM, and XGBoost, to determine which algorithm performs best in terms of accuracy and error reduction.  
   - Understand how each feature contributes to the prediction of delivery times and identify critical variables influencing delays.  

2. *Classification Problem:*  
   - Optionally, classify deliveries as "On-time" or "Delayed" based on a threshold applied to the delivery time.  
   - Explore whether classification models can provide meaningful insights into delivery performance by identifying patterns in the categorical and numerical variables.  
   - Investigate the effectiveness of classification techniques in prioritizing deliveries or predicting potential delays under specific conditions like high traffic or adverse weather.  

3. *Dimensionality Reduction:*  
   - Analyze whether the dataset contains high-dimensional features that may introduce redundancy or noise, potentially impacting model performance.  
   - Apply dimensionality reduction techniques, such as feature selection or PCA, to retain only the most relevant information while improving computational efficiency and reducing overfitting.  

*Objective:*

The primary goal of this project is to develop accurate and reliable machine learning models that can predict delivery times while gaining valuable insights into the underlying factors affecting efficiency and delays. By understanding these factors, the project aims to provide actionable recommendations for optimizing operations in food delivery services. The objectives include:  

- *Accuracy:* Achieve high accuracy in predicting delivery times by leveraging advanced machine learning algorithms and tailored feature engineering techniques.  
- *Insights:* Gain a deeper understanding of how features such as traffic level, vehicle type, and weather conditions impact delivery performance.  
- *Operational Improvements:* Use the findings to propose strategies for reducing delays, improving resource allocation, and enhancing customer satisfaction.  
- *Scalability:* Ensure that the proposed models and methodologies can be applied to larger datasets and diverse geographic regions, making them adaptable for real-world use cases.  

By addressing these research questions and objectives, the project aims to enhance the operational efficiency of food delivery systems, provide more accurate delivery time estimates, and improve overall service quality.

In [2]:
# Load the data
df = pd.read_csv("datasets/Food_Delivery_Times.csv", index_col= False)

In [3]:
# Display the data
df

,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68
...,...,...,...,...,...,...,...,...,...
995,107,8.50,Clear,High,Evening,Car,13,3.0,54
996,271,16.28,Rainy,Low,Morning,Scooter,8,9.0,71
997,861,15.62,Snowy,High,Evening,Scooter,26,2.0,81
998,436,14.17,Clear,Low,Afternoon,Bike,8,0.0,55


In [371]:
# Display the data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Order_ID                1000 non-null   int64  
 1   Distance_km             1000 non-null   float64
 2   Weather                 970 non-null    object 
 3   Traffic_Level           970 non-null    object 
 4   Time_of_Day             970 non-null    object 
 5   Vehicle_Type            1000 non-null   object 
 6   Preparation_Time_min    1000 non-null   int64  
 7   Courier_Experience_yrs  970 non-null    float64
 8   Delivery_Time_min       1000 non-null   int64  
 9   Preparation_Ratio       1000 non-null   float64
 10  Traffic_Vehicle         1000 non-null   object 
dtypes: float64(3), int64(3), object(5)
memory usage: 86.1+ KB


In [5]:
# Display the data insights
df.describe()

,Order_ID,Distance_km,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
count,1000.000000,1000.000000,1000.000000,970.000000,1000.000000
mean,500.500000,10.059970,16.982000,4.579381,56.732000
std,288.819436,5.696656,7.204553,2.914394,22.070915
min,1.000000,0.590000,5.000000,0.000000,8.000000
25%,250.750000,5.105000,11.000000,2.000000,41.000000
50%,500.500000,10.190000,17.000000,5.000000,55.500000
75%,750.250000,15.017500,23.000000,7.000000,71.000000
max,1000.000000,19.990000,29.000000,9.000000,153.000000


## Data Exploration

Descriptive Approaches:

1. Basic Statistics:

Calculated measures of central tendency (mean, median) and variability (standard deviation) for numerical variables.

Analyzed the frequency distribution of categorical variables.

2. Visualizations:

Histograms: Showed the skewed distribution of Delivery_Time_min.

Boxplots: Identified outliers in numerical variables like Distance_km and Preparation_Time_min.

Bar Charts: Explored the distribution of categorical variables such as Weather and Traffic_Level.

Scatterplots: Examined relationships between Distance_km and Delivery_Time_min, color-coded by Traffic_Level and Weather.

Heatmap: Showed correlations between numerical variables, with strong correlations between Distance_km and Delivery_Time_min.

Interpretation:

Delivery times are influenced by multiple factors, with longer distances and adverse weather/traffic conditions leading to delays.

Outliers were retained to ensure the model learns from real-world extremes.

### Missing Values

There are four columns with missing values; three of them are categorical, and one is numerical.

In [6]:
# Display the missing values
df.isnull().sum()

Order_ID                   0
Distance_km                0
Weather                   30
Traffic_Level             30
Time_of_Day               30
Vehicle_Type               0
Preparation_Time_min       0
Courier_Experience_yrs    30
Delivery_Time_min          0
dtype: int64

### Target Variable : Delivery_Time_min

In [8]:
# Visualize the delivery time distribution
fig = px.histogram(df, x='Delivery_Time_min', 
             labels={'Delivery_Time_min': 'Delivery Time (min)', 'count': 'Count'}, 
             title='Delivery Time Distribution')
fig.show()

In [9]:
# Visualize the outlier values of delivery time 
fig = px.box(df, y= "Delivery_Time_min",
             labels={'value': 'Value', 'variable': 'Variable'},
             title='Boxplot of Multiple Variables')
fig.show()

### Outlier Values in Numerical Variables

In [10]:
#Visualize the outlier values of distance, preparation time and courier experience
fig = px.box(df, y=['Distance_km', 'Preparation_Time_min', 'Courier_Experience_yrs'],
             labels={'value': 'Value', 'variable': 'Variable'},
             title='Boxplot of Multiple Variables')

fig.show()

### Distribution of Categorical Variables

In [11]:
weather_count = df["Weather"].value_counts().reset_index()
weather_count.columns = ["Weather", "Count"]

fig = px.bar(weather_count, x = "Weather", y = "Count", title="Weather Distribution")
fig.show()

In [12]:
traffic_count = df["Traffic_Level"].value_counts().reset_index()
traffic_count.columns = ["Traffic_Level", "Count"]

fig = px.bar(traffic_count, x = "Traffic_Level", y = "Count", title="Traffic Level Distribution")
fig.show()

In [13]:
time_count = df["Time_of_Day"].value_counts().reset_index()
time_count.columns = ["Time_of_Day", "Count"]

fig = px.bar(time_count, x = "Time_of_Day", y = "Count", title="Time of Day Distribution")
fig.show()

In [14]:
vehicle_count = df["Vehicle_Type"].value_counts().reset_index()
vehicle_count.columns = ["Vehicle_Type", "Count"]

fig = px.bar(vehicle_count, x = "Vehicle_Type", y = "Count", title="Vehicle Type Distribution")
fig.show()

### Bivariate Analysis

In [15]:
# Visualize the relationship between distance and delivery time with traffic level
fig = px.scatter(df, x="Distance_km", y="Delivery_Time_min", color="Traffic_Level",labels= {"Distance_km":"Distance (km)", "Delivery_Time_min":"Delivery Time (min)"}, title = "Distance vs Delivery Time with Traffic Level")
fig.show()

In [64]:
# Visualize the relationship between distance and delivery time with weather
fig = px.scatter(df, x="Distance_km", y="Delivery_Time_min", color="Weather",labels= {"Distance_km":"Distance (km)", "Delivery_Time_min":"Delivery Time (min)"}, title = "Distance vs Delivery Time with Weather")
fig.show()

In [65]:
# Visualize the relationship between distance and delivery time with time of day
fig = px.scatter(df, x="Distance_km", y="Delivery_Time_min", color="Time_of_Day",labels= {"Distance_km":"Distance (km)", "Delivery_Time_min":"Delivery Time (min)"}, title = "Distance vs Delivery Time with Time of Day")
fig.show()

In [66]:
# Visualize the relationship between distance and delivery time with vehicle type
fig = px.scatter(df, x="Distance_km", y="Delivery_Time_min", color="Vehicle_Type",labels= {"Distance_km":"Distance (km)", "Delivery_Time_min":"Delivery Time (min)"}, title = "Distance vs Delivery Time with Vehicle Level")
fig.show()

In [67]:
# Visualize the relationship between courier experience and delivery time with traffic level
fig = px.bar(df, x="Courier_Experience_yrs", y="Delivery_Time_min", color="Traffic_Level",labels= {"Distance_km":"Distance (km)", "Delivery_Time_min":"Delivery Time (min)"}, title = "Courier Experience Years vs Delivery Time with Traffic Level")
fig.show()

In [68]:
# Visualize the relationship between courier experience and delivery time with weather
fig = px.bar(df, x="Courier_Experience_yrs", y="Delivery_Time_min", color="Weather",labels= {"Distance_km":"Distance (km)", "Delivery_Time_min":"Delivery Time (min)"}, title = "Courier Experience Years vs Delivery Time with Weather")
fig.show()

In [69]:
# Visualize the outlier values of delivery time with traffic level
fig = px.box(df, x="Traffic_Level", y="Delivery_Time_min",labels= {"Distance_km":"Distance (km)", "Delivery_Time_min":"Delivery Time (min)"}, title = "Traffic Level vs Delivery Time")
fig.show()

In [22]:
# Visualize the delivery time vs. traffic level

time_df = df.groupby("Time_of_Day")["Delivery_Time_min"].mean().reset_index()

fig = px.line(time_df, x="Time_of_Day", y="Delivery_Time_min",labels= {"Time_of_Day":"Time of Day", "Delivery_Time_min":"Mean of Delivery Time"}, title = "Time of Day vs Delivery Time")
fig.show()

In [23]:
# Visualize the delivery time vs. vehicle type with traffic level
vehicle_df = df.groupby(["Vehicle_Type", "Traffic_Level"])["Delivery_Time_min"].mean().reset_index()

fig = px.line(vehicle_df, x="Vehicle_Type", y="Delivery_Time_min",color= "Traffic_Level", labels= {"Vehicle_Type":"Vehicle_Type", "Delivery_Time_min":"Mean of Delivery Time"}, title = "Vehicle Type vs Delivery Time")
fig.show()

### Correlation of Continuous Variables

In [24]:
#Visualize the correlation matrix of continuous variables
corr_matrix = df[["Distance_km", "Preparation_Time_min", "Courier_Experience_yrs", "Delivery_Time_min"]].corr()

fig = px.imshow(corr_matrix,labels=dict(x="Variables", y="Variables", color="Correlation"),
                x=corr_matrix.columns,
                y=corr_matrix.columns,
                title="Correlation Heatmap of Continuous Variables", text_auto=True)
fig.show()

## Methodologies

Data Pre-processing:

1. Missing Value Imputation:

Categorical variables imputed with the most frequent category.

Numerical variables imputed with the mean.

2. Feature Scaling:

StandardScaler was applied to numerical variables to normalize their ranges.

Feature Engineering:

1. Preparation_Ratio:

Captures the efficiency of preparation relative to delivery time.

2. Traffic_Vehicle:

Reflects interactions between traffic levels and vehicle types.

Train-Test Split:

The dataset was split into 80% training data and 20% test data to evaluate model performance on unseen data.

Modeling Techniques:

Linear Regression: Used as a baseline model.

LightGBM and XGBoost: Applied to capture non-linear relationships and improve accuracy.



### Feature Engineering


In [25]:
#df['Distance_per_minute'] = df['Distance_km'] / df['Delivery_Time_min']

In [26]:
# Calculate the preparation ratio as a new feature
df['Preparation_Ratio'] = df['Preparation_Time_min'] / df['Delivery_Time_min']

In [27]:
# Calculate the traffic_vehicle as a new feature. Scooter is fast in high traffic, medium in medium traffic and slow in low traffic. Car is slow in high traffic. Bike is fast in high traffic, medium in medium traffic and slow in low traffic.
conditions = [
    ((df["Traffic_Level"] == "High") & (df["Vehicle_Type"] == "Scooter")),
    ((df["Traffic_Level"] == "Medium") & (df["Vehicle_Type"] == "Scooter")),
    ((df["Traffic_Level"] == "Low") & (df["Vehicle_Type"] == "Car")),
    ((df["Traffic_Level"] == "High") & (df["Vehicle_Type"] == "Bike")),
    ((df["Traffic_Level"] == "Medium") & (df["Vehicle_Type"] == "Car")),
    ((df["Traffic_Level"] == "Low") & (df["Vehicle_Type"] == "Scooter"))
]

choices = ["Fast", "Fast", "Fast", "Medium", "Medium", "Medium"]

df["Traffic_Vehicle"] = np.select(conditions, choices, default="Slow")


In [28]:
df

,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min,Preparation_Ratio,Traffic_Vehicle
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43,0.279070,Medium
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84,0.238095,Slow
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59,0.474576,Medium
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37,0.135135,Fast
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68,0.235294,Slow
...,...,...,...,...,...,...,...,...,...,...,...
995,107,8.50,Clear,High,Evening,Car,13,3.0,54,0.240741,Slow
996,271,16.28,Rainy,Low,Morning,Scooter,8,9.0,71,0.112676,Medium
997,861,15.62,Snowy,High,Evening,Scooter,26,2.0,81,0.320988,Fast
998,436,14.17,Clear,Low,Afternoon,Bike,8,0.0,55,0.145455,Slow


### Missing Value Imputation and Encoding

In [30]:
# Making the pipeline for the imputer and one hot encoder
ohe_pipe = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(sparse_output=False)
)

In [31]:
# Making the pipeline for the imputer and ordinal encoder
ord_pipe = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OrdinalEncoder()
)

In [32]:
# Making the pipeline for the imputer and standard scaler
cont_pipe = make_pipeline(
    SimpleImputer(strategy="mean"),
    StandardScaler()
)

In [33]:
# Selecting the columns for the column transformer
ohe_columns = ["Time_of_Day","Vehicle_Type"]
ord_columns = ["Weather", "Traffic_Level", "Traffic_Vehicle"]
cont_columns = ["Distance_km","Preparation_Time_min", "Courier_Experience_yrs",  "Preparation_Ratio"]

In [34]:
# Making the column transformer
transformers = [("ohe", ohe_pipe, ohe_columns),
                ("ord", ord_pipe, ord_columns),
                ("cont", cont_pipe, cont_columns)]

In [35]:
col_transform = ColumnTransformer(transformers)

### Train Test Split

In [37]:
# Splitting the data into X and y 
X = df.drop(columns=["Delivery_Time_min"])
y = df[["Delivery_Time_min"]]

In [38]:
# Splitting the data into train and test sets 
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=900)

In [39]:
# Control the data shape
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(800, 10)
(200, 10)
(800, 1)
(200, 1)


In [40]:
# Ravelling the y_train
y_train = y_train.values.ravel()

In [41]:
# Ravelling the y_test
y_test = y_test.values.ravel()

### Linear Regression

In [42]:
# Making the linear regression pipeline
linear_pipe = make_pipeline(col_transform,
                            LinearRegression())

In [43]:
# Fitting the linear regression pipeline
linear_pipe.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('ohe',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(sparse_output=False))]),
                                                  ['Time_of_Day',
                                                   'Vehicle_Type']),
                                                 ('ord',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ordinalencoder',
                                                                   OrdinalEncoder())]),
                                                  ['Weather', 'Traffic_Level',
                                                   'Traffic_Vehicle']),
                                                 ('cont',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer()),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  ['Distance_km',
                                                   'Preparation_Time_min',
                                                   'Courier_Experience_yrs',
                                                   'Preparation_Ratio'])])),
                ('linearregression', LinearRegression())])

In [44]:
# Predicting the delivery time
y_pred_lin = linear_pipe.predict(X_test)

In [46]:
# Evaluating the model
r2 = r2_score(y_test, y_pred_lin)
mae = mean_absolute_error(y_test, y_pred_lin)
mse = mean_squared_error(y_test, y_pred_lin)
rmse = root_mean_squared_error(y_test, y_pred_lin)

print(f"R²: {r2}")
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")

R²: 0.8071693101057636
MAE: 6.42171875
MSE: 92.98843505859375
RMSE: 9.643051128071123


### LightGBM

In [48]:
# Arrange the parameters for the lightgbm model
params = {
    "objective": ["regression"],
    "metric": ["rmse"],
    "boosting_type": ["gbdt"],
    "num_leaves": [31],
    "learning_rate": [0.05],
    "feature_fraction": [0.9],
    "lambda_l1": [0.1],
    "lambda_l2": [0.1],
    "min_gain_to_split": [0.01]
}

In [49]:
# Arrange the grid parameters for the lightgbm model
param_grid = {
    "num_leaves": [31, 50, 70],
    "learning_rate": [0.05, 0.1, 0.2],
    "feature_fraction": [0.8, 0.9, 1.0]
}

In [50]:
# Arrange the lightgbm model
gbm = lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt')

In [51]:
# Arrange the grid search for the lightgbm model
grid_search = GridSearchCV(estimator=gbm, param_grid=param_grid, cv=3, scoring="neg_mean_squared_error", verbose=1)

In [52]:
# Making the encoding pipeline to encode the data
encode_pipe = make_pipeline(col_transform)

In [53]:
# Fitting the grid search
grid_search.fit(encode_pipe.fit_transform(X_train), y_train)

Fitting 3 folds for each of 27 candidates, totalling 81 fits
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001809 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 418
[LightGBM] [Info] Number of data points in the train set: 533, number of used features: 14
[LightGBM] [Info] Start training from score 56.981238
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

GridSearchCV(cv=3,
             estimator=LGBMRegressor(metric='rmse', objective='regression'),
             param_grid={'feature_fraction': [0.8, 0.9, 1.0],
                         'learning_rate': [0.05, 0.1, 0.2],
                         'num_leaves': [31, 50, 70]},
             scoring='neg_mean_squared_error', verbose=1)

In [54]:
print("Best Hyperparameters for LightGBM: ", grid_search.best_params_)
print("Best Score for LightGBM: ", grid_search.best_score_)

Best Hyperparameters for LightGBM:  {'feature_fraction': 1.0, 'learning_rate': 0.2, 'num_leaves': 31}
Best Score for LightGBM:  -23.749356362725592


In [55]:
# Selecting the best parameters
best_params = grid_search.best_params_

In [56]:
# Updating the parameters
params.update(best_params)

In [57]:
# Selecting the categorical columns
categories = ["Time_of_Day", "Vehicle_Type", "Weather", "Traffic_Level", "Traffic_Vehicle"]

In [58]:
# Converting the columns to categorical
for col in categories:
    X_train[col] = X_train[col].astype("category")
    X_test[col] = X_test[col].astype("category")

In [59]:
# Making the lightgbm dataset
train_data_lgb =lgb.Dataset(X_train, label=y_train, categorical_feature=categories)
test_data_lgb = lgb.Dataset(X_test, label=y_test, reference=train_data_lgb, categorical_feature=categories)

In [60]:
# Training the lightgbm model
model = lgb.train(params, train_data_lgb, num_boost_round=1000, valid_sets=[train_data_lgb, test_data_lgb])

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000373 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 823
[LightGBM] [Info] Number of data points in the train set: 800, number of used features: 10
[LightGBM] [Info] Start training from score 56.655000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

In [61]:
# Predicting the delivery time
y_pred_lgb = model.predict(X_test, num_iteration=model.best_iteration)

In [62]:
# Evaluating the model
r2 = r2_score(y_test, y_pred_lgb)
mae = mean_absolute_error(y_test, y_pred_lgb)
mse = mean_squared_error(y_test, y_pred_lgb)
rmse = root_mean_squared_error(y_test, y_pred_lgb)

print(f"R²: {r2}")
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")

R²: 0.9632410041022043
MAE: 2.738899569925385
MSE: 17.726231777400596
RMSE: 4.210253172601452


### XGBoost

In [71]:
# Making the xgboost model
xgbr = xgb.XGBRegressor(random_state=900, enable_categorical=True)

In [72]:
# Converting the columns to categorical
for col in categories:
    X_train[col] = X_train[col].astype("category")
    X_test[col] = X_test[col].astype("category")

In [73]:
# Evaluatiing the cross validation score for the xgboost model to get the RMSE
xgb_cv = cross_val_score(xgbr, X_train, y_train, cv=5, scoring="neg_mean_squared_error")
print("XGB CV RMSE: ", np.sqrt(-xgb_cv.mean()))

XGB CV RMSE:  5.76966614903495


In [74]:
# Arranging the grid search parameters for the xgboost model
param_grid = {
    "n_estimators": [100, 200, 300, 500, 1000],
    "learning_rate": [0.01, 0.1, 0.15, 0.2],
    "max_depth": [3, 5, 7, 10, 15],
    "subsample": [0.8, 1],
    "colsample_bytree": [0.8, 1],
}

In [75]:
# Arranging the grid search for the xgboost model
grid_search = GridSearchCV(estimator=xgbr, param_grid=param_grid, cv=3, scoring="neg_mean_squared_error", verbose=1)

In [ ]:
# Fitting the grid search
grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 400 candidates, totalling 1200 fits


In [367]:
print("Best Hyperparameters for XGBoost: ", grid_search.best_params_)
print("Best Score for XGBoost: ", grid_search.best_score_)

Best Hyperparameters for XGBoost:  {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 1000, 'subsample': 0.8}
Best Score for XGBoost:  -17.00870731623505


In [368]:
#selecting the best model
best_model = grid_search.best_estimator_

In [369]:
# Fitting the best model and predicting the delivery time
y_pred_xgb = best_model.predict(X_test)

In [370]:
# Evaluating the model
r2 = r2_score(y_test, y_pred_xgb)
mae = mean_absolute_error(y_test, y_pred_xgb)
mse = mean_squared_error(y_test, y_pred_xgb)
rmse = root_mean_squared_error(y_test, y_pred_xgb)

print(f"R²: {r2}")
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")

R²: 0.9704702496528625
MAE: 2.5354404067993164
MSE: 14.240080673187876
RMSE: 3.7736031419835174


## Results


The project evaluated three machine learning models for predicting food delivery times: Linear Regression, LightGBM, and XGBoost. The results demonstrated the following performances:
<br>
<br>

|Rank| Model| R^2|MAE|MSE|RMSE|
|----|------|--------|---------|----------|---------|
|1| Linear Regression|0.807 | 6.42|92.99|9.64
|2| LightGBM|0.963 | 2.74|17.73|4.21
|3| XGBoost|0.970 | 2.54|14.24|3.77

<br>

XGBoost emerged as the best-performing model, achieving the highest R² score (0.970) and the lowest error metrics. It effectively captured complex patterns and non-linear relationships within the data.

LightGBM also delivered strong results with a slightly lower R² score (0.963), demonstrating excellent efficiency and speed.

Linear Regression provided a baseline performance but struggled to handle non-linear interactions and complex dependencies, resulting in moderate accuracy.

These results highlight the effectiveness of advanced machine learning models, particularly XGBoost, in optimizing predictions and uncovering intricate relationships within the dataset.


## Conclusion

This project aimed to predict food delivery times using a dataset that included a variety of features, such as distance, preparation time, traffic level, weather conditions, and vehicle type. Through comprehensive data exploration, preprocessing, and feature engineering, we developed and evaluated multiple machine learning models, including Linear Regression, LightGBM, and XGBoost.

The results demonstrated that advanced models like XGBoost and LightGBM significantly outperformed Linear Regression by capturing complex relationships and non-linear interactions within the data. XGBoost emerged as the top-performing model, achieving an R² score of 0.970 and the lowest error metrics (MAE: 2.54, RMSE: 3.77). LightGBM followed closely with an R² score of 0.963, showcasing its efficiency and ability to handle categorical and numerical variables effectively. Linear Regression, while interpretable, provided a moderate baseline performance with an R² score of 0.807.

One of the key strengths of this project was the application of feature engineering. Features such as Preparation_Ratio and Traffic_Vehicle added significant value to the dataset by capturing critical relationships, such as preparation efficiency and the interaction between traffic levels and vehicle types. These features improved model performance and provided actionable insights into the factors influencing delivery times.

Additionally, robust preprocessing steps were implemented to handle missing values, encode categorical variables, and scale numerical features. These ensured the dataset was clean and well-prepared for modeling, further enhancing the reliability of the results.

*Key Insights:*
1. Distance and traffic level are major contributors to delivery times, with longer distances and high traffic causing significant delays.
2. Weather conditions, particularly rainy or snowy weather, also impact delivery times, highlighting the need for context-aware predictions.
3. Scooters tend to perform better under high traffic conditions compared to cars and bikes, emphasizing the role of vehicle choice in operational efficiency.

*Future Work:*
1. *Real-Time Data Integration:* Incorporating live traffic and weather data into the models would enable dynamic predictions, making the solution more practical for real-world applications.
2. *Modeling Enhancements:* Exploring ensemble modeling techniques, such as stacking or blending, could combine the strengths of multiple algorithms to achieve even higher accuracy.
3. *Feature Enrichment:* Adding more detailed operational data, such as order volume or courier availability, could provide a more comprehensive view of factors influencing delivery times.
4. *Scalability Testing:* Expanding the approach to larger datasets and more diverse geographic areas would validate the robustness of the models.

This project provides a solid framework for using machine learning to optimize food delivery operations. By leveraging advanced techniques and carefully engineered features, it offers valuable insights for improving efficiency, reducing delays, and enhancing customer satisfaction. The learnings from this work can be extended to other industries and applications where accurate time prediction is critical.



## Work Distribution

- We did the all project together so
- Zeynep Aslı Üretme %50
- Ebru Özcan %50

## References

- All the external sources (including images, data set access etc should be cited here).

## Additional Folders

If this document involves datasets and images, please keep them in datasets and img folders, respectively. Please, be sure that your project is REPRODUCIBLE. Otherwise, it may not be graded.